In [7]:
!pip install dlib

ERROR: Could not find a version that satisfies the requirement dlib (from versions: none)
ERROR: No matching distribution found for dlib


In [8]:
"""
╔══════════════════════════════════════════════════════════════╗
║          SMART ATTENDANCE — Face Recognition                 ║
║          + Anti-Spoof Liveness Detection                     ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
"""

import dlib
import numpy as np
import cv2
import os
import pandas as pd
import time
import logging
import sqlite3
import datetime
import random
from scipy.spatial import distance as dist
from PIL import Image, ImageDraw, ImageFont

# ──────────────────────────────────────────────────────────────
#  Dlib models
# ──────────────────────────────────────────────────────────────
detector        = dlib.get_frontal_face_detector()
predictor       = dlib.shape_predictor('shape_predictor_68_face_landmarks.dat')
face_reco_model = dlib.face_recognition_model_v1(
                      "dlib_face_recognition_resnet_model_v1.dat")

# ──────────────────────────────────────────────────────────────
#  PIL font  (falls back to default if no TTF available)
# ──────────────────────────────────────────────────────────────
try:
    _PIL_FONT_LG = ImageFont.truetype("arial.ttf", 22)
    _PIL_FONT_SM = ImageFont.truetype("arial.ttf", 17)
except Exception:
    _PIL_FONT_LG = ImageFont.load_default()
    _PIL_FONT_SM = ImageFont.load_default()

# ──────────────────────────────────────────────────────────────
#  Database
# ──────────────────────────────────────────────────────────────
_conn = sqlite3.connect("attendance.db")
_conn.execute(
    "CREATE TABLE IF NOT EXISTS attendance "
    "(name TEXT, time TEXT, date DATE, UNIQUE(name, date))")
_conn.commit()
_conn.close()


# ══════════════════════════════════════════════════════════════
#  PIL text helper  — draws Unicode/emoji on an OpenCV frame
# ══════════════════════════════════════════════════════════════
def put_pil_text(frame_bgr, text, pos, color_rgb=(255, 255, 255),
                 font=None, bg_color=None):
    """
    Render 'text' (with emoji / arrows) onto an OpenCV BGR frame using PIL.
    pos = (x, y)  — top-left corner of the text.
    """
    if font is None:
        font = _PIL_FONT_LG

    # convert frame slice to PIL
    img_pil = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    draw    = ImageDraw.Draw(img_pil)

    x, y = pos
    if bg_color is not None:
        bbox = draw.textbbox((x, y), text, font=font)
        pad  = 4
        draw.rectangle([bbox[0]-pad, bbox[1]-pad,
                         bbox[2]+pad, bbox[3]+pad],
                        fill=bg_color)

    draw.text((x, y), text, font=font, fill=color_rgb)
    frame_bgr[:] = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)


def put_banner(frame_bgr, text, color_rgb=(0, 255, 80), bg_rgb=(0, 40, 0)):
    """Full-width banner at the top of the frame."""
    h, w = frame_bgr.shape[:2]
    img_pil = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    draw    = ImageDraw.Draw(img_pil)
    draw.rectangle([0, 0, w, 58], fill=bg_rgb)
    draw.text((12, 12), text, font=_PIL_FONT_LG, fill=color_rgb)
    frame_bgr[:] = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)


# ══════════════════════════════════════════════════════════════
#  Liveness helpers
# ══════════════════════════════════════════════════════════════
BLINK_EAR_THRESHOLD = 0.21
BLINK_CONSEC_FRAMES = 2
LIVENESS_TIMEOUT    = 6   # seconds per attempt

STATE_BLINK        = "blink"
STATE_DOUBLE_BLINK = "double_blink"
STATE_HEAD         = "head"
STATE_NOD          = "nod"
STATE_PASSED       = "passed"
STATE_FAILED       = "failed"

ALL_CHALLENGES = [
    ("blink",        None),
    ("blink",        None),
    ("double_blink", None),
    ("head",        "left"),
    ("head",        "right"),
    ("head",        "up"),
    ("head",        "down"),
    ("nod",          None),
]

CHALLENGE_LABELS = {
    "blink":        "👁  Blink once!",
    "double_blink": "👁👁  Blink TWICE!",
    "head_left":    "⬅  Turn head LEFT",
    "head_right":   "➡  Turn head RIGHT",
    "head_up":      "⬆  Look UP",
    "head_down":    "⬇  Look DOWN",
    "nod":          "↕  NOD your head",
}

# BGR colours for cv2 rectangles
COL_CHALLENGE = (0, 165, 255)   # orange
COL_PASSED    = (0, 220,  60)   # green
COL_FAILED    = (0,   0, 220)   # red
COL_UNKNOWN   = (0,   0, 180)   # dark red
COL_NEUTRAL   = (180, 180, 180) # grey


def eye_aspect_ratio(pts):
    A = dist.euclidean(pts[1], pts[5])
    B = dist.euclidean(pts[2], pts[4])
    C = dist.euclidean(pts[0], pts[3])
    return (A + B) / (2.0 * C)


def get_ears(shape):
    c = np.array([[shape.part(i).x, shape.part(i).y] for i in range(68)])
    return c[42:48], c[36:42]   # left_eye, right_eye


def head_direction(shape):
    c  = np.array([[shape.part(i).x, shape.part(i).y] for i in range(68)])
    cx, cy   = int(c[:,0].mean()), int(c[:,1].mean())
    nx, ny   = c[30]
    dx, dy   = nx - cx, ny - cy
    HT, VT   = 15, 12
    if abs(dx) > abs(dy):
        if dx >  HT: return "right"
        if dx < -HT: return "left"
    else:
        if dy < -VT: return "up"
        if dy >  VT: return "down"
    return "center"


def pick_challenges(n=2):
    pool = list({(t, v) for t, v in ALL_CHALLENGES})   # deduplicate
    random.shuffle(pool)
    return pool[:n]


# ══════════════════════════════════════════════════════════════
class LivenessSession:
    def __init__(self, name):
        self.name          = name
        self.start_time    = time.time()
        self.challenges    = pick_challenges(2)
        self.idx           = 0
        self.blink_cnt     = 0
        self.blink_done    = 0
        self.nod_down      = False
        self.state         = self._cur_state()

    def _cur_state(self):
        if self.idx >= len(self.challenges):
            return STATE_PASSED
        return self.challenges[self.idx][0]

    def _advance(self):
        self.idx       += 1
        self.blink_cnt  = 0
        self.blink_done = 0
        self.nod_down   = False
        self.state      = self._cur_state()

    def timed_out(self):
        return (time.time() - self.start_time) > LIVENESS_TIMEOUT

    def remaining(self):
        return max(0.0, LIVENESS_TIMEOUT - (time.time() - self.start_time))

    def progress(self):
        done  = self.idx
        total = len(self.challenges)
        return f"{'█'*done}{'░'*(total-done)}  {done}/{total}"

    def label(self):
        if self.idx >= len(self.challenges):
            return None
        ctype, cval = self.challenges[self.idx]
        return f"head_{cval}" if ctype == "head" else ctype

    def update(self, shape):
        if self.state in (STATE_PASSED, STATE_FAILED):
            return self.state
        if self.timed_out():
            self.state = STATE_FAILED
            return self.state

        l, r  = get_ears(shape)
        ear   = (eye_aspect_ratio(l) + eye_aspect_ratio(r)) / 2.0
        direc = head_direction(shape)
        ctype, cval = self.challenges[self.idx]

        if ctype == "blink":
            if ear < BLINK_EAR_THRESHOLD:
                self.blink_cnt += 1
            else:
                if self.blink_cnt >= BLINK_CONSEC_FRAMES:
                    self._advance()
                self.blink_cnt = 0

        elif ctype == "double_blink":
            if ear < BLINK_EAR_THRESHOLD:
                self.blink_cnt += 1
            else:
                if self.blink_cnt >= BLINK_CONSEC_FRAMES:
                    self.blink_done += 1
                    if self.blink_done >= 2:
                        self._advance()
                self.blink_cnt = 0

        elif ctype == "head":
            if direc == cval:
                self._advance()

        elif ctype == "nod":
            if not self.nod_down and direc == "down":
                self.nod_down = True
            elif self.nod_down and direc in ("up", "center"):
                self._advance()

        return self.state


# ══════════════════════════════════════════════════════════════
#  Draw liveness UI on frame
# ══════════════════════════════════════════════════════════════
def draw_liveness_box(frame, session, rect):
    x1, y1, x2, y2 = rect.left(), rect.top(), rect.right(), rect.bottom()
    state = session.state

    if state == STATE_PASSED:
        cv2.rectangle(frame, (x1,y1),(x2,y2), COL_PASSED, 3)
        put_pil_text(frame, "✅  Identity Confirmed!",
                     (x1, y1-32), (0,220,60), bg_color=(0,40,0))
        return

    if state == STATE_FAILED:
        cv2.rectangle(frame, (x1,y1),(x2,y2), COL_FAILED, 3)
        put_pil_text(frame, "❌  Spoof Detected — Try Again",
                     (x1, y1-32), (220,60,60), bg_color=(40,0,0))
        return

    # active challenge
    lbl = session.label()
    challenge_text = CHALLENGE_LABELS.get(lbl, "")
    cv2.rectangle(frame, (x1,y1),(x2,y2), COL_CHALLENGE, 2)
    put_pil_text(frame, challenge_text,
                 (x1, y1-52), (255,165,0), _PIL_FONT_LG, bg_color=(30,20,0))
    put_pil_text(frame, f"{session.progress()}   ⏱ {session.remaining():.1f}s",
                 (x1, y1-28), (200,200,200), _PIL_FONT_SM, bg_color=(20,20,20))


def draw_unknown_box(frame, rect, pos):
    x1, y1, x2, y2 = rect.left(), rect.top(), rect.right(), rect.bottom()
    cv2.rectangle(frame, (x1,y1),(x2,y2), COL_UNKNOWN, 2)
    put_pil_text(frame, "🚫  Not Registered",
                 pos, (180,60,60), _PIL_FONT_SM, bg_color=(30,0,0))


# ══════════════════════════════════════════════════════════════
#  Main recognizer
# ══════════════════════════════════════════════════════════════
class FaceRecognizer:

    def __init__(self):
        self.cv_font = cv2.FONT_HERSHEY_SIMPLEX

        # FPS
        self.fps = self.fps_show = 0.0
        self.frame_start_time = self.start_time = time.time()
        self.frame_cnt = 0

        # Known faces
        self.known_features : list = []
        self.known_names    : list = []

        # Frame state
        self.cur_names     : list = []
        self.last_names    : list = []
        self.cur_centroids : list = []
        self.last_centroids: list = []
        self.cur_cnt  = 0
        self.last_cnt = 0
        self.cur_positions   : list = []
        self.cur_features    : list = []
        self.cur_e_dists     : list = []

        self.reclassify_cnt      = 0
        self.reclassify_interval = 10

        # Liveness state
        # name -> LivenessSession  (pending)
        self.sessions  : dict[str, LivenessSession] = {}
        # names that passed liveness THIS session (reset when face disappears)
        self.passed    : set[str] = set()
        # names that already had attendance recorded today
        self.recorded  : set[str] = set()
        # names that failed (shown message already)
        self.failed_msg: set[str] = set()

        # track which names were visible last frame
        self.prev_visible: set[str] = set()

        # welcome banner  name -> expiry timestamp
        self.banners: dict[str, float] = {}

    # ── database ──────────────────────────────────────────────
    def load_db(self):
        if not os.path.exists("All_Features.csv"):
            logging.warning("All_Features.csv not found!")
            return False
        df = pd.read_csv("All_Features.csv", header=None)
        for i in range(len(df)):
            self.known_names.append(df.iloc[i, 0])
            feats = [0.0 if df.iloc[i, j] == '' else float(df.iloc[i, j])
                     for j in range(1, 129)]
            self.known_features.append(feats)
        logging.info("Loaded %d faces from DB", len(self.known_names))
        return True

    def save_attendance(self, name):
        # طبع مرة واحدة بس لكل شخص في الـ session
        if name in self.recorded:
            return

        today   = datetime.datetime.now().strftime('%Y-%m-%d')
        now_str = datetime.datetime.now().strftime('%H:%M:%S')
        display = name.replace("_", " ")

        conn = sqlite3.connect("attendance.db")
        cur  = conn.cursor()
        cur.execute("SELECT 1 FROM attendance WHERE name=? AND date=?",
                    (name, today))
        already = cur.fetchone()

        if not already:
            cur.execute(
                "INSERT INTO attendance (name,time,date) VALUES (?,?,?)",
                (name, now_str, today))
            conn.commit()
            print("\n" + "═"*55)
            print(f"  🟢  IDENTITY CONFIRMED — {display}")
            print(f"  🕐  {now_str}    📅  {today}")
            print(f"  🚪  Door OPEN — Welcome aboard, {display}!")
            print(f"  ✅  Attendance recorded successfully.")
            print("═"*55 + "\n")
        else:
            print("\n" + "═"*55)
            print(f"  🔵  WELCOME BACK — {display}")
            print(f"  🕐  {now_str}    📅  {today}")
            print(f"  🚪  Door OPEN — Have a great day, {display}!")
            print(f"  ℹ️   Already signed in today.")
            print("═"*55 + "\n")

        conn.close()
        self.recorded.add(name)                 # لا يطبع تاني في نفس الـ session
        self.banners[name] = time.time() + 3.0  # بانر على الشاشة 3 ثواني

    # ── face math ─────────────────────────────────────────────
    @staticmethod
    def euclidean(a, b):
        return float(np.sqrt(np.sum(np.square(np.array(a) - np.array(b)))))

    def centroid_track(self):
        for i, cc in enumerate(self.cur_centroids):
            dists = [self.euclidean(cc, lc) for lc in self.last_centroids]
            self.cur_names[i] = self.last_names[dists.index(min(dists))]

    # ── liveness management ───────────────────────────────────
    def _liveness_for(self, name, shape, frame, rect):
        """
        Run liveness check for 'name'.
        Returns True ONLY on the exact frame liveness passes.
        After that, just returns True silently (already in self.passed).
        """
        # Already confirmed this session
        if name in self.passed:
            # Still draw green box so user sees confirmation
            draw_liveness_box(frame, _PassedDummy(), rect)
            return True

        # Create / re-create session
        if name not in self.sessions or \
                self.sessions[name].state == STATE_FAILED:
            self.sessions[name] = LivenessSession(name)
            self.failed_msg.discard(name)

        s     = self.sessions[name]
        state = s.update(shape)

        draw_liveness_box(frame, s, rect)

        if state == STATE_PASSED:
            self.passed.add(name)
            del self.sessions[name]
            return True

        if state == STATE_FAILED and name not in self.failed_msg:
            self.failed_msg.add(name)
            display = name.replace("_", " ")
            print("\n" + "═"*55)
            print(f"  🔴  LIVENESS FAILED — {display}")
            print("  🤖  Possible spoof attempt detected!")
            print("  📸  Photo / screen / video ≠ real human.")
            print("  🚪  Door LOCKED.")
            print("═"*55 + "\n")

        return False

    def _handle_gone_faces(self, visible: set):
        """Reset liveness for faces that left the frame."""
        gone = self.prev_visible - visible
        for name in gone:
            if name == "unknown":
                continue
            self.passed.discard(name)
            self.sessions.pop(name, None)
            self.failed_msg.discard(name)
            self.recorded.discard(name)   # يطبع جملة الترحيب تاني لو رجع
        # reset unknown guards when no faces visible
        if not visible or visible == {"unknown"}:
            self.recorded = {n for n in self.recorded if not n.startswith("unknown_")}
        self.prev_visible = set(visible)

    # ── HUD + banners ─────────────────────────────────────────
    def draw_hud(self, frame):
        put_pil_text(frame, "🎯  Smart Attendance System",
                     (16, 8), (255,255,255), _PIL_FONT_LG)
        cv2.putText(frame,
                    f"FPS:{self.fps_show:.0f}  Faces:{self.cur_cnt}  Frame:{self.frame_cnt}",
                    (16, 48), self.cv_font, 0.5, (0,220,0), 1, cv2.LINE_AA)
        cv2.putText(frame, "Q = quit",
                    (16, frame.shape[0]-10), self.cv_font,
                    0.5, (180,180,180), 1, cv2.LINE_AA)

        now = time.time()
        for name, exp in list(self.banners.items()):
            if now < exp:
                put_banner(frame,
                           f"🚪  ACCESS GRANTED — Welcome, {name.replace('_',' ')}!",
                           (0,255,80), (0,50,0))
            else:
                del self.banners[name]

    # ── core recognition for ONE face ─────────────────────────
    def _recognize_face_k(self, frame, k, face_rect):
        """
        Compute recognition for face k.
        Returns (name, distance).
        """
        shape = predictor(frame, face_rect)
        feat  = face_reco_model.compute_face_descriptor(frame, shape)

        best_d, best_i = 999.0, -1
        for i, kf in enumerate(self.known_features):
            if float(kf[0]) == 0.0:
                continue
            d = self.euclidean(feat, kf)
            if d < best_d:
                best_d, best_i = d, i

        if best_d < 0.4 and best_i >= 0:
            return self.known_names[best_i], best_d
        return "unknown", best_d

    # ── main loop ─────────────────────────────────────────────
    def process(self, stream):
        if not self.load_db():
            return

        while stream.isOpened():
            self.frame_cnt += 1
            ok, frame = stream.read()
            if not ok or frame is None:
                continue

            key   = cv2.waitKey(1)
            faces = detector(frame, 0)

            self.last_cnt       = self.cur_cnt
            self.cur_cnt        = len(faces)
            self.last_names     = self.cur_names[:]
            self.last_centroids = self.cur_centroids[:]
            self.cur_centroids  = []

            # ── build centroid list ───────────────────────────
            self.cur_positions = []
            for k, d in enumerate(faces):
                cx = (d.left()  + d.right())  / 2
                cy = (d.top()   + d.bottom()) / 2
                self.cur_centroids.append([cx, cy])
                self.cur_positions.append((
                    d.left(),
                    int(d.bottom() + (d.bottom()-d.top())/4)))

            # ── decide whether to re-classify ─────────────────
            need_reclassify = (
                self.cur_cnt != self.last_cnt or
                self.reclassify_cnt >= self.reclassify_interval
            )

            if need_reclassify:
                self.reclassify_cnt = 0
                # fresh recognition for every face
                self.cur_names = []
                for k, face_rect in enumerate(faces):
                    name, _ = self._recognize_face_k(frame, k, face_rect)
                    self.cur_names.append(name)
            else:
                # inherit names via centroid tracker
                if self.cur_cnt > 0:
                    if self.cur_cnt == self.last_cnt and self.last_centroids:
                        self.cur_names = ["unknown"] * self.cur_cnt
                        self.centroid_track()
                    else:
                        # centroid list changed size — re-do
                        self.cur_names = []
                        for k, face_rect in enumerate(faces):
                            name, _ = self._recognize_face_k(frame, k, face_rect)
                            self.cur_names.append(name)
                else:
                    self.cur_names = []

                if "unknown" in self.cur_names:
                    self.reclassify_cnt += 1

            # ── draw boxes + run liveness ─────────────────────
            visible_now = set()
            for k, face_rect in enumerate(faces):
                name = self.cur_names[k] if k < len(self.cur_names) else "unknown"
                visible_now.add(name)

                # draw name label
                cv2.putText(frame, name, self.cur_positions[k],
                            self.cv_font, 0.75, (0,255,255), 1, cv2.LINE_AA)

                if name == "unknown":
                    draw_unknown_box(frame, face_rect, self.cur_positions[k])
                    # طبع في الـ terminal مرة واحدة لكل وجه غريب
                    uid = f"unknown_{k}"
                    if uid not in self.recorded:
                        self.recorded.add(uid)
                        print("\n" + "═"*55)
                        print("  🔴  ACCESS DENIED")
                        print("  👤  Face not registered in the system.")
                        print("  🚪  Door remains LOCKED.")
                        print("═"*55 + "\n")
                else:
                    shape_lv = predictor(frame, face_rect)
                    liveness_just_passed = self._liveness_for(
                        name, shape_lv, frame, face_rect)

                    if liveness_just_passed:
                        # attendance called every frame after pass,
                        # but UNIQUE constraint + self.recorded prevent duplicates
                        self.save_attendance(name)

            self._handle_gone_faces(visible_now)
            self.draw_hud(frame)

            if key == ord('q'):
                break

            # FPS
            now = time.time()
            if str(self.start_time).split('.')[0] != str(now).split('.')[0]:
                self.fps_show = self.fps
            self.start_time = now
            ft = now - self.frame_start_time
            self.fps = 1.0 / ft if ft > 0 else 0
            self.frame_start_time = now

            cv2.namedWindow("Smart Attendance", 1)
            cv2.imshow("Smart Attendance", frame)

    def run(self):
        cap = cv2.VideoCapture(0)
        self.process(cap)
        cap.release()
        cv2.destroyAllWindows()


# ── dummy passed session for green-box display ────────────────
class _PassedDummy:
    state = STATE_PASSED


# ══════════════════════════════════════════════════════════════
if __name__ == '__main__':
    logging.basicConfig(level=logging.INFO)
    FaceRecognizer().run()


ModuleNotFoundError: No module named 'dlib'